# SDR Query Notebook Quickstart

This notebook is organized in three stages:

1. Plain TypeScript in a notebook cell
2. Notebook UI helpers with local sample data
3. Connecting to SDR and starting Gremlin and Cypher queries


## 1. Plain TypeScript Runs Inline

A notebook cell is just TypeScript. You can define values, transform data, use
`console.log(...)`, and render results in the same UI.

In [3]:
import { ui } from "@sdr-notebook/mod";

const numbers = [3, 5, 8, 13];
const summary = {
  count: numbers.length,
  total: numbers.reduce((sum, value) => sum + value, 0),
  doubled: numbers.map((value) => value * 2),
};

console.log("Numbers:", numbers);
console.log("Summary:", summary);

ui.json(summary);


Numbers: [ 3, 5, 8, 13 ]
Summary: { count: 4, total: 29, doubled: [ 6, 10, 16, 26 ] }


{
  "count": 4,
  "total": 29,
  "doubled": [
    6,
    10,
    16,
    26
  ]
}

## 2. Notebook UI Helpers

Before connecting to Neptune, here are the main notebook display helpers using
ordinary TypeScript data only.

In [ ]:
import { bar, pie } from "@sdr-notebook/mod";

const sampleStudies = [
  { id: "STUDY-101", phase: "Phase 1", status: "Draft", sites: 8 },
  { id: "STUDY-204", phase: "Phase 2", status: "Active", sites: 23 },
  { id: "STUDY-319", phase: "Phase 3", status: "Closed", sites: 41 },
];

const statusCounts = {
  Draft: 1,
  Active: 1,
  Closed: 1,
};

const sampleGraph = {
  vertices: [
    { id: "study-101", label: "Study", properties: { name: "STUDY-101" } },
    {
      id: "version-1",
      label: "StudyVersion",
      properties: { name: "Version 1" },
    },
    { id: "site-a", label: "Site", properties: { name: "Site A" } },
  ],
  edges: [
    { id: "edge-1", label: "has_version", outV: "study-101", inV: "version-1" },
    { id: "edge-2", label: "runs_at", outV: "study-101", inV: "site-a" },
  ],
};

### `ui.table(...)` and `ui.json(...)`

- `ui.table(...)` is best for arrays of records.
- `ui.json(...)` is best when you want to inspect the raw shape of an object.


In [ ]:
ui.table(sampleStudies);


In [ ]:
ui.json({
  firstStudy: sampleStudies[0],
  availableStatuses: Object.keys(statusCounts),
});


### `ui.graph(...)`

`ui.graph(...)` renders graph-shaped data. In practice you will usually pass
Gremlin `.path()` results, but it also accepts a plain `{ vertices, edges }`
object.

In [ ]:
ui.graph(sampleGraph, {
  width: 900,
  height: 500,
  labelProperty: "name",
  showEdgeLabels: true,
});


### `ui.md(...)`, `ui.html(...)`, and `ui.withLoader(...)`

- `ui.md` emits inline markdown.
- `ui.html(...)` emits lightweight custom HTML.
- `ui.withLoader(...)` shows progress while async work is running.


In [ ]:
ui.md`### Sample Notebook Note\n\nThis markdown block was rendered from a TypeScript template literal.`;

ui.html(
  `<div style="padding:12px;border:1px solid #cbd5e1;border-radius:8px;background:#f8fafc">Custom HTML works too.</div>`,
  "Custom HTML works too.",
);

const delayedSummary = await ui.withLoader(
  "Preparing local sample data...",
  async () => {
    await new Promise((resolve) => setTimeout(resolve, 300));
    return sampleStudies.map((study) => ({ id: study.id, sites: study.sites }));
  },
  { successMessage: "Local sample data ready." },
);

ui.table(delayedSummary);


### Chart Helpers

Charts are not `ui.*` methods, but they are useful notebook helpers when you
want a quick visual summary.

In [ ]:
pie(statusCounts, { title: "Study Statuses", donut: true });

bar(
  sampleStudies.map((study) => ({ label: study.id, value: study.sites })),
  { title: "Sites Per Study" },
);


## 3. Connect to SDR and Start Querying

Now that the notebook basics are out of the way, the rest of the quickstart
moves into the real SDR workflow.

Prerequisite:

```bash
aws sso login --profile dsoadev
```

Then connect the notebook session to Neptune through the SDR REST query API.

In [4]:
import {
  connectNotebook,
  sampleStudies as sampleStudyAliases,
} from "@sdr-notebook/mod";

const session = await connectNotebook({ profile: "dsoadev" });
ui.md`# Connected\n\nProfile: ${"dsoadev"}`;

# Connected

Profile: dsoadev

### Using the AWS SDK With the Same Profile

The notebook session can also create AWS SDK v3 clients using the same resolved
credentials and region as your Neptune connection.

In [ ]:
import { ListBucketsCommand, S3Client } from "npm:@aws-sdk/client-s3";

const s3 = session.aws(S3Client);
const buckets = await s3.send(new ListBucketsCommand({}));

ui.table(
  (buckets.Buckets ?? []).map((bucket) => ({
    name: bucket.Name,
    createdAt: bucket.CreationDate?.toISOString(),
  })),
);


### Gremlin Basics

Gremlin is a traversal language. You move through the graph step by step.

Common ideas:

- `g.V()` starts from all vertices
- `.hasLabel("Study")` filters by vertex label
- `.has("name", "X")` filters by property
- `.out("edge")` follows outgoing edges with a given label
- `.elementMap()` returns vertex properties
- `.path()` returns visited vertices and edges, which is what graph rendering
  usually needs

Starter queries:

```gremlin
g.V().hasLabel("Study").count()
g.V().hasLabel("Study").limit(5).elementMap()
g.V().hasLabel("Study").limit(1).out("has_version").elementMap()
```

In [5]:
const gremlinStudies = await session.gremlin<Array<Record<string, unknown>>>(
  `g.V().hasLabel("Study").limit(5).elementMap()`,
);

ui.table(
  gremlinStudies.map((study) => ({
    id: study.id,
    name: study.name,
    description: study.description,
  })),
);

id,name,description
QU8-US-AKBZ,QU8-US-AKBZ,Prospective registry quantifying COPD exacerbations and biomarker changes.
CRD-09-2036,CRD-09-2036,Prospective registry quantifying COPD exacerbations and biomarker changes.
NWS-UW-IGSB,NWS-UW-IGSB,Testing DSoA in Dev
SHA-MC-BXCC,SHA-MC-BXCC,Testing SDR in Dev
E2E-TA-PRAD,E2E-TA-PRAD,


### Gremlin SDK Input

You can also pass a fluent Gremlin traversal object to `session.gremlin(...)`
instead of a raw string.

In [1]:
const g = session.g();

const gremlinSdkStudies = await session.gremlin<Array<Record<string, unknown>>>(
  g.V().hasLabel("Study").limit(3).elementMap(),
);

ui.table(
  gremlinSdkStudies.map((study) => ({
    id: study.id,
    name: study.name,
    description: study.description,
  })),
);


ReferenceError: session is not defined

### Cypher Basics

Cypher is pattern-matching oriented. Instead of walking the graph step by step,
you describe the shape you want.

Common ideas:

- `(s:Study)` means a node with label `Study`
- `-[:has_version]->` means an outgoing edge with that type
- `RETURN ...` controls the output
- `LIMIT n` keeps results small

Starter queries:

```cypher
MATCH (s:Study)
RETURN s
LIMIT 5

MATCH (s:Study)-[:has_version]->(v:StudyVersion)
RETURN s, v
LIMIT 5
```

In [6]:
const cypherStudies = await session.cypher(
  `MATCH (s:Study) RETURN s LIMIT 5`,
);

ui.json(cypherStudies);

[
  {
    "s": {
      "~id": "bacd630c-1479-45fc-878f-647ce09361b6",
      "~entityType": "node",
      "~labels": [
        "Study"
      ],
      "~properties": {
        "createdAt": "2025-11-27T16:14:23.447Z",
        "systemName": "CDISC USDM E2J",
        "instanceType": "Study",
        "name": "QU8-US-AKBZ",
        "description": "Prospective registry quantifying COPD exacerbations and biomarker changes.",
        "id": "QU8-US-AKBZ",
        "label": "BRIGHT-AIR: COPD Real-World Registry",
        "systemVersion": "0.62.0",
        "usdmVersion": "4.0.0",
        "updatedAt": "2025-11-27T16:14:23.447Z",
        "studyId": "f3b6d747-b88c-4b35-8900-81883e018ff3"
      }
    }
  },
  {
    "s": {
      "~id": "68cd6477-4c7b-4810-e05c-36c0f6e56d9b",
      "~entityType": "node",
      "~labels": [
        "Study"
      ],
      "~properties": {
        "createdAt": "2025-11-28T05:27:51.262Z",
        "systemName": "CDISC USDM E2J",
        "instanceType": "Study",
        "name": "CRD-09-2036",
        "description": "Prospective registry quantifying COPD exacerbations and biomarker changes.",
        "id": "CRD-09-2036",
        "label": "BRIGHT-AIR: COPD Real-World Registry",
        "systemVersion": "0.62.0",
        "usdmVersion": "4.0.0",
        "updatedAt": "2026-03-17T12:01:57.205Z",
        "studyId": [
          "642e8cfc-293c-4514-846b-c8509ff9823b",
          "7080f1a5-d84e-41e7-be4b-a3eab7840858",
          "f55081dc-042b-4eca-826c-a4b4558ceb83",
          "3291c53b-aaef-4c92-a0f5-94838680eff0",
          "c9643830-6b5b-4b82-a949-6de82db1cb21",
          "61ad9a0e-4daf-4911-bae3-3d010f41c5a6",
          "51797108-644f-4c21-b1b7-32e86b5e558c",
          "3b01d0b6-017b-4f05-a2b1-7cc3972874a3",
          "420e2228-3b93-41f0-b270-04142feb3da5",
          "a9783b80-1933-4dc4-9656-e21341dcb35f",
          "b1c57c10-aa52-4ee6-98db-82bf29118546",
          "e366ad9e-f779-47e1-ac26-12cc3583f3ed",
          "9b6b846b-cf6b-4338-ad62-14220d14e980",
          "ef7f4641-59fa-4ea8-b7af-6998a8681dab",
          "f8a25a5d-58d0-4643-b96b-fb22a43abfae",
          "4dbc7638-7150-4399-ac4c-876492d69287",
          "fee95862-d9cf-40f6-874f-43dbd77e4a65",
          "8e38edb9-78da-4883-a197-61e2b29dee2f",
          "0f6d66d7-0c4a-4789-a423-c2ca3941b5c2",
          "3301cf47-5775-428c-aa5f-76e955fe6400",
          "ffda342f-2649-419b-9de0-7735f5313302",
          "bc285c55-a167-4ef1-bf3c-89e370d5c9b6",
          "ffc92d5e-9ab7-4868-b294-1a49669be2ac",
          "3ffaa07a-5cbf-43f2-817e-b4a75b2964d8",
          "7d5698a1-da12-4bfb-87ea-ed64155b4126",
          "f6c289b0-5038-4da3-8eef-3c5c2e4c3caf",
          "ae2dca64-c0bf-4b7d-befe-e643efcf417e",
          "831fbb5d-247c-4fe4-b6b4-f4025bacdb3b",
          "3724a994-3009-4ac4-b908-202b87c7de73",
          "a956104a-9dad-4729-a4ae-4042db9a4e31",
          "045b5840-143c-47a5-b67f-cde5a48a1cf9",
          "e174984f-045f-42fe-bbf8-f09fc3b7d2f9",
          "1517839c-9d2e-4a79-9e78-3c1d45f2e0aa",
          "b24f0940-5887-4e87-a000-2e25fd6183b4",
          "3ae935a1-3592-4a91-9671-1232132a9af2",
          "01c2049b-ab6f-4b1b-bbf7-de566ed2afc0",
          "67e1b29c-724f-4563-903e-62bbb5090b0d",
          "8a2a5487-ff74-4f44-84c2-ade01336ac83",
          "fb8afc08-ae4e-4044-a1e1-83d09a9ca583",
          "7529f661-bc41-4089-863c-543e37048aec",
          "1e1e1ad4-bfcd-481a-a735-f698c8af252d",
          "8a1400ce-a734-41dc-80e3-bf182c5038d0",
          "f70ff33a-0507-4f66-8f63-61f7d87cb6f6",
          "2cb09131-811e-4b1e-a021-3778b82b7612",
          "7642f6db-70d8-437f-8f68-0e54310e8019",
          "ea60ce5c-44d1-4eda-8267-e15a0ef6f5ed",
          "7c542ead-07f1-452a-a287-8e5a1296a365",
          "c1421098-bccf-4683-9af0-64e085eb72f3",
          "6b5dc42d-8561-4f7f-9829-13dcd1bf17d5",
          "c8d19152-1a6f-4379-9d0a-82847d59e75f",
          "fe3b75ed-35bb-41f3-9cf6-2c51871a199a",
          "83270a39-aff4-421b-bbb5-6f67ed110a8a",
          "b8fbaf46-bcaf-4eac-8ccb-c726b66c40b9",
          

### Cypher Builder Input

If you prefer typed query construction, `session.cypher(...)` also accepts a
`@neo4j/cypher-builder` clause.

In [ ]:
import Cypher from "@neo4j/cypher-builder";

const s = new Cypher.NamedNode("s");
const cypherBuilderStudies = await session.cypher(
  new Cypher.Match(new Cypher.Pattern(s, { labels: ["Study"] }))
    .return(s)
    .limit(3),
);

ui.table(cypherBuilderStudies);

s
"{ ""~id"": ""bacd630c-1479-45fc-878f-647ce09361b6"", ""~entityType"": ""node"", ""~labels"": [ ""Study"" ], ""~properties"": { ""createdAt"": ""2025-11-27T16:14:23.447Z"", ""systemName"": ""CDISC USDM E2J"", ""instanceType"": ""Study"", ""name"": ""QU8-US-AKBZ"", ""description"": ""Prospective registry quantifying COPD exacerbations and biomarker changes."", ""id"": ""QU8-US-AKBZ"", ""label"": ""BRIGHT-AIR: COPD Real-World Registry"", ""systemVersion"": ""0.62.0"", ""usdmVersion"": ""4.0.0"", ""updatedAt"": ""2025-11-27T16:14:23.447Z"", ""studyId"": ""f3b6d747-b88c-4b35-8900-81883e018ff3"" } }"
"{ ""~id"": ""e2cd64cb-0860-804d-d2bb-ba26a99a5310"", ""~entityType"": ""node"", ""~labels"": [ ""Study"" ], ""~properties"": { ""createdAt"": ""2025-11-28T08:30:46.440Z"", ""systemName"": ""CDISC USDM E2J"", ""instanceType"": ""Study"", ""name"": ""NWS-UW-IGSB"", ""description"": ""Testing DSoA in Dev"", ""id"": ""NWS-UW-IGSB"", ""label"": ""DSOA01"", ""systemVersion"": ""0.62.0"", ""usdmVersion"": ""4.0.0"", ""updatedAt"": ""2025-11-28T08:30:46.440Z"", ""studyId"": ""23201ef6-93b9-4808-9583-facc396e9fa8"" } }"


### Rendering Query Results as a Graph

To render a graph from live Neptune data, return paths rather than simple
property maps. In Gremlin that usually means calling `.path()`.

In [ ]:
const sampledStudies = await sampleStudyAliases(session, {
  minVersions: 2,
  limit: 1,
});
const studyAlias = sampledStudies[0]?.alias;

if (!studyAlias) {
  ui.md`_No sampled study was found._`;
} else {
  const paths = await session.gremlin(
    `g.V().has("name", "${studyAlias}").hasLabel("Study").outE().inV().path().limit(20)`,
  );

  ui.graph(paths, {
    width: 1200,
    height: 800,
    linkDistance: 180,
    chargeStrength: -450,
    nodeRadius: 28,
    labelMaxChars: 20,
    showNodeType: true,
    showEdgeLabels: true,
  });
}

### Notebook Tips

- Cells share state, so run the notebook from top to bottom the first time.
- If AWS credentials expire, run `aws sso login --profile dsoadev` again.
- Keep graph queries small with `limit(...)`, especially when using `.path()`.
- If outputs look stale or variables seem missing, restart the kernel and run
  all cells again.